# Exploring ToponymExtractor predictions

Inspecting prediction outputs amongst overlapping png sections.

In [ ]:
# Imports
from typing import Final
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from os import getenv
import numpy as np
import pandas as pd
import geopandas as gp
import matplotlib.pyplot as plt
from edina import get_png_overlaps
from outputs import get_intersecting_polygon_pairs, get_intersecting_png_masks

# Paths
PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

## Get overlapping regions

In [ ]:
ctrl_points = gp\
    .read_file(LOCAL_DIR.joinpath("outputs/pngs/control-points.gpkg"))
ctrl_points.head()

In [ ]:
png_overlaps = get_png_overlaps(ctrl_points = ctrl_points)
png_overlaps.head()

In [ ]:
tif_overlaps = png_overlaps[["tiff_filename", "geometry"]]\
    .dissolve("tiff_filename", as_index = False)
tif_overlaps["geometry"] = tif_overlaps.geometry.buffer(0)
tif_overlaps.head()

## Get example predictions

In [ ]:
# Load predictions
pred_idx = 15
predictions =\
    [*LOCAL_DIR.glob("outputs/toponym-extractor/extracted/*.gpkg")]
pred_ex = gp.read_file(predictions[pred_idx]) # example file
pred_ex["geometry"] = pred_ex.geometry.buffer(0)
pred_ex.head()

## Determine which predictions are contained within overlaps

In [ ]:
selection =\
    tif_overlaps["tiff_filename"] == f"{predictions[pred_idx].stem}.tif"
tif_overlap = tif_overlaps.loc[selection, "geometry"].iloc[0]
tif_overlap

In [ ]:
# default to all predictions belonging to a single png
pred_ex["png_overlap"] = "disjoint"
# update predictions that intersect with png overlap area
selection = pred_ex.geometry.intersects(tif_overlap)
pred_ex.loc[selection, "png_overlap"] = "intersect"
# update predictions that are contained entirely within png overlap area
selection = pred_ex.geometry.within(tif_overlap)
pred_ex.loc[selection, "png_overlap"] = "subset"

pred_ex["png_overlap"].value_counts()

In [ ]:
pred_overlap = [
    "png_filename",
    "groupid",
    "wordid",
    "png_overlap",
    "word",
    "score",
    "geometry"
]
pred_overlap = pred_ex[pred_overlap]
pred_overlap = gp.sjoin(pred_overlap, pred_overlap)
pred_overlap["same_png"] =\
    (pred_overlap.png_filename_left == pred_overlap.png_filename_right)

# Exclude records that join the same prediction instance to itself
pred_overlap = pred_overlap[(pred_overlap.index != pred_overlap.index_right)]

# Select columns
pred_overlap = pred_overlap[[
    "png_filename_left",
    "groupid_left",
    "wordid_left",
    "png_overlap_left",
    "word_left",
    "score_left",
    "index_right",
    "png_overlap_right",
    "word_right",
    "score_right",
    "same_png",
    "geometry"
]]

pred_overlap["intersection_area"] = pred_overlap.geometry\
    .intersection(pred_ex.loc[pred_overlap.index_right, "geometry"], align=0)\
    .area
pred_overlap["union_area"] = pred_overlap.geometry\
    .union(pred_ex.loc[pred_overlap.index_right, "geometry"], align=0)\
    .area
pred_overlap["iou"] =\
    pred_overlap["intersection_area"] / pred_overlap["union_area"]
pred_overlap.head()

In [ ]:
pred_overlap\
    .loc[pred_overlap.index < pred_overlap.index_right]\
    .groupby(["png_overlap_left", "png_overlap_right"])\
    .agg(count = ("index_right", "count"))

In [ ]:
# check "disjoint", "disjoint" overlaps should have same png names - 
# block should always return False:
(
    (pred_overlap.png_overlap_left == "disjoint")
    & (pred_overlap.png_overlap_right == "disjoint")
    & (~pred_overlap.same_png)
).any()

In [ ]:
overlap_summary = pred_overlap[[
    "png_filename_left", "groupid_left", "wordid_left", "same_png", "iou"
]]
overlap_summary = overlap_summary.groupby(
    by = ["png_filename_left", "groupid_left", "wordid_left", "same_png"],
    as_index = False)
overlap_summary = overlap_summary.max()
overlap_summary

### Inspecting IoU scores for overlaping predictions

In [ ]:
# get bin widths
bins = np.histogram(overlap_summary.iou.array, bins = 100)[1]

fig, ax = plt.subplots(figsize = (10, 6))
hist = ax.hist(
    overlap_summary.loc[overlap_summary.same_png, "iou"],
    bins = bins,
    alpha = .5,
    label = "Polygons belonging to the same PNG"
)
hist = ax.hist(
    overlap_summary.loc[~overlap_summary.same_png, "iou"],
    bins = bins,
    alpha = .5,
    label = "Polygons belonging to different PNGs"
)
_ = ax.legend()
_ = ax.set_xlabel("IoU")
_ = ax.set_ylabel("Counts")
_ = ax.set_title(
    "Intersection over Union scores for overlapping polygons in overlapping "\
    "PNG spaces"
)

## (Subset, Subset) - Inspecting IoU scores for overlaping predictions

In [ ]:
selection = (
    (pred_overlap.png_overlap_left == "subset")
    & (pred_overlap.png_overlap_right == "subset")
    & (pred_overlap.png_filename_left != pred_overlap.png_filename_right)
    & (pred_overlap.word_left == pred_overlap.word_right)
)
fig, ax = plt.subplots(figsize = (10, 6))
hist = ax.hist(
    pred_overlap.loc[selection, "iou"],
    bins = 100,
    alpha = .5,
    label = "Different PNG (subset) - Same Word"
)

selection = (
    (pred_overlap.png_overlap_left == "subset")
    & (pred_overlap.png_overlap_right == "subset")
    & (pred_overlap.png_filename_left != pred_overlap.png_filename_right)
    & (pred_overlap.word_left != pred_overlap.word_right)
)
hist = ax.hist(
    pred_overlap.loc[selection, "iou"],
    bins = 100,
    alpha = .5,
    label = "Different PNG (subset) - Different Word"
)

selection = (
    pred_overlap.png_filename_left == pred_overlap.png_filename_right
)
hist = ax.hist(
    pred_overlap.loc[selection, "iou"],
    bins = 100,
    alpha = .5,
    label = "Same PNG"
)
_ = ax.legend()
_ = ax.set_xlabel("IoU")
_ = ax.set_ylabel("Counts")
_ = ax.set_title(
    "Intersection over Union scores for overlapping polygons in overlapping "\
    "PNG spaces"
)

Set IoU at .8 for non-maximal suppresion.

Inspection of IoU distributions suggests two types of intersecting prediction. Those for the same text instance and those for a different text instance. We make use of DeepSolo's bipartite matching here, which should minimize the likelihood of overlapping word predictions belonging to the same text instance. The higher frequency of predictions of different words across different PNGs in the low IoU range should represent genuinely different text instance predictions, matching the distribution shape of overlapping predictions within the same PNG. The high IoU score matching the shape of predictions of the same word label across different pngs suggest these predictions are genuinely of the same text instance.

## Intersect - Subset inspection
Looking at the score distribution between insterct and subset.

In [ ]:
selection = (
    (pred_overlap.png_overlap_left == "intersect")
    & (pred_overlap.png_overlap_right == "subset")
    # | ((pred_overlap.png_overlap_left == "subset")
    #     & (pred_overlap.png_overlap_right == "intersect"))
    & (pred_overlap.png_filename_left != pred_overlap.png_filename_right)
    & (pred_overlap.word_left != pred_overlap.word_right)
)
intersect_scores = pred_overlap.loc[selection, "score_left"].tolist()
subset_scores = pred_overlap.loc[selection, "score_right"].tolist()
iou_scores = pred_overlap.loc[selection, "iou"].tolist()

selection = (
    (pred_overlap.png_overlap_left == "subset")
    & (pred_overlap.png_overlap_right == "intersect")
    & (pred_overlap.png_filename_left != pred_overlap.png_filename_right)
    & (pred_overlap.word_left != pred_overlap.word_right)
)
intersect_scores += pred_overlap.loc[selection, "score_right"].tolist()
subset_scores += pred_overlap.loc[selection, "score_left"].tolist()
iou_scores += pred_overlap.loc[selection, "iou"].tolist()

fig, ax = plt.subplots(figsize = (10, 10))
scatter = ax.scatter(
    intersect_scores,
    subset_scores,
    s = [el * 200 for el in iou_scores],
    c = iou_scores,
    cmap = "viridis"
)
cbar = plt.colorbar(scatter)
cbar.set_label("IoU")
# Set axis labels
_ = ax.set_xlabel("Intersecting prediction scores")
_ = ax.set_ylabel("Subset prediction scores")
# Set axis ranges
selection = min(*intersect_scores, *subset_scores) - .1
_ = ax.set_ylim(selection, 1.), ax.set_xlim(selection, 1.), ax.set_aspect(1)
# Add y=x line
_ = ax.axline(
    (selection, selection),
    (1., 1.),
    linestyle = "--",
    linewidth = 1,
    color = "navy"
)
# Show grid
_ = ax.grid(True)


In [ ]:
selection = (
    (
        (pred_overlap.png_overlap_left == "intersect")
            & (pred_overlap.png_overlap_right == "subset")
        | ((pred_overlap.png_overlap_left == "subset")
            & (pred_overlap.png_overlap_right == "intersect"))
    )
    & (pred_overlap.png_filename_left != pred_overlap.png_filename_right)
    & (pred_overlap.word_left != pred_overlap.word_right)
    & (pred_overlap.iou > .1)
)
pred_overlap.loc[selection]

Intersect <-> Subset selection:
- If word predictions are the same, then drop the "subset" prediction, otherwise;
- If IoU is less than .1, then keep both both instances, otherwise;
- If the "subset" prediction string is contained within the "intersect" prediction, then drop the "subset" prediction, otherwise;
- Return the merged polygon.

## Applying Selection

### Applying NMS

In [ ]:
temp_overlap = pred_overlap.copy()
discounted_idx = set()

# Drop "subset" word instances where the predicted words are the same - 
# keep highest scoring instance
temp_overlap = temp_overlap[(
    (temp_overlap.png_overlap_left == "subset")
    & (temp_overlap.png_overlap_right == "subset")
    & (temp_overlap.word_left == temp_overlap.word_right)
)]

for tup in temp_overlap.itertuples(index = True):
    discounted_idx.add((
        tup.index_right
        if tup.score_left >= tup.score_right
        else tup.Index
    ))
selection = [el for el in pred_ex.index if el not in discounted_idx]
temp_overlap = pred_ex.loc[selection].copy()

# Apply NMS to "subset" - "subset" instances
temp_overlap = temp_overlap[(temp_overlap["png_overlap"] == "subset")]
temp_overlap = get_intersecting_polygon_pairs(temp_overlap)
temp_overlap = temp_overlap.loc[(temp_overlap.iou > .8)]
for tup in temp_overlap.itertuples(index = True):
    discounted_idx.add((
        tup.index_right
        if tup.score_left >= tup.score_right
        else tup.Index
    ))
selection = [el for el in pred_ex.index if el not in discounted_idx]
temp_overlap = pred_ex.loc[selection].copy()

# Apply NMS to "intersect" - "subset" instances
temp_overlap = get_intersecting_png_masks(
    temp_overlap[temp_overlap["png_overlap"] == "intersect"],
    temp_overlap[temp_overlap["png_overlap"] == "subset"]
)
# Drop subset instances when word instances are the same
selection = (temp_overlap.word_left == temp_overlap.word_right)
discounted_idx.update(temp_overlap.loc[selection, "index_right"].tolist())
temp_overlap = temp_overlap[~selection]
# Keep both instances when IoU < .1
temp_overlap = temp_overlap[temp_overlap.iou >= .1]
# if "intersection" string starts or ends with the "subset" string, then
# drop the "subset" string
selection = [
    (
        tup.word_left.startswith(tup.word_right)
        or tup.word_left.endswith(tup.word_right)
    )
    for tup in temp_overlap.itertuples()
]
selection = pd.Series(selection, index = temp_overlap.index)
discounted_idx.update(temp_overlap.loc[selection, "index_right"].tolist())

selection = [el for el in pred_ex.index if el not in discounted_idx]
temp_overlap = pred_ex.loc[selection].copy()

# tup = next(pred_iter)
# try:
#     while 1:

# except StopIteration as e:
#     print("All records complete")

In [ ]:
temp_overlap

## Intersect - Intersect predictions

In [ ]:
temp_overlap = get_intersecting_png_masks(
    pred_ex[pred_ex["png_overlap"] == "intersect"],
    pred_ex[pred_ex["png_overlap"] == "intersect"]
)
temp_overlap = temp_overlap[temp_overlap.index < temp_overlap.index_right]
temp_overlap = temp_overlap[temp_overlap.iou > .1]
temp_overlap = temp_overlap[(
    (temp_overlap.iou < .8)
    | (temp_overlap.word_left != temp_overlap.word_right)
)]

temp_overlap

Predictions with small mask IoU scores (less than or equal to 0.1) will be treated as separate.

Predictions with high mask IoU (greater than or equal to .8) with the same word label will have the prediction with the lower IoU score suppressed.

Otherwise, predictions will be treated as ambiguous, so image snippets will be extracted for the regions the word predictions belong to.

### Retrieve image snippets for ambiguous predictions

In [ ]:
check = [{1, 2}, {3, 4}, {5, 6}]
test = {0, 7}
clique = next((el for el in check if not test.isdisjoint(el)), set())
bool(clique)
# clique.update(test)
# check

In [ ]:
from rasterio import open as open_raster
from rasterio.transform import AffineTransformer
from numpy import hstack, floor as np_floor, ceil as np_ceil
import matplotlib.pyplot as plt

# Get all instances of "intersect" - "intersect" overlapping predictions
bbox = temp_overlap\
    .index.union(temp_overlap.index_right).unique().sort_values()
bbox = pred_ex.loc[bbox]
# Retrieve the groups the overlapping predictions belong to
bbox["key"] = bbox["png_filename"] + bbox["groupid"].astype("string")
selection = pred_ex["png_filename"] + pred_ex["groupid"].astype("string")
bbox = pred_ex[selection.isin(set(bbox["key"]))]
bbox["key"] = bbox["png_filename"] + bbox["groupid"].astype("string")
# Create grouped geometries
bbox = bbox[["key", "png_filename", "groupid", "geometry"]]\
    .dissolve(["key", "png_filename", "groupid"], as_index = True)

# Add key fields for left and right predictions
selection = temp_overlap.copy()
selection["key_left"] =\
    selection.png_filename_left + selection.groupid_left.astype("string")
selection["key_right"] =\
    selection.png_filename_right + selection.groupid_right.astype("string")

bbox = bbox.loc[selection.key_left.tolist(), "geometry"]\
    .union(bbox.loc[selection.key_right.tolist(), "geometry"], align = False)\
    .reset_index(drop = False, name = "geometry")\
    .dissolve("key", as_index = True)\
    .buffer(5)\
    .convex_hull\
    .sort_index()\
    .reset_index(drop = False, name = "geometry")

tiff_fn = LOCAL_DIR.joinpath(f"data/tiffs/{predictions[pred_idx].stem}.tif")
with open_raster(tiff_fn, mode = "r") as src:
    tiff_height = src.height # image height
    tiff_width = src.width # image width
    tiff_transformer = AffineTransformer(src.transform) # transformer
    tiff_crs = src.read_crs() # coordinate reference system
    data = (-src.read() + 1) * 255 # image array

bbox = bbox.to_crs(tiff_crs).bounds
bbox[["min_row", "min_col"]] = hstack([
    el.reshape((-1, 1))
    for el
    in tiff_transformer.rowcol(bbox["minx"], bbox["maxy"], op = np_floor)
])
bbox[["max_row", "max_col"]] = hstack([
    el.reshape((-1, 1))
    for el
    in tiff_transformer.rowcol(bbox["maxx"], bbox["miny"], op = np_ceil)
])
assert ((bbox.min_row <= bbox.max_row) & (bbox.min_col <= bbox.max_col)).all(), "Incoherent row/col ranges"

bbox[["min_row", "max_row"]] = bbox[["min_row", "max_row"]]\
    .clip(0, tiff_height - 1)
bbox[["min_col", "max_col"]] = bbox[["min_col", "max_col"]]\
    .clip(0, tiff_width - 1)
bbox[["min_row", "min_col", "max_row", "max_col"]] =\
    bbox[["min_row", "min_col", "max_row", "max_col"]].astype("int64")

# bbox
temp = bbox.iloc[6]
temp = data[
    0,
    int(temp.min_row): int(temp.max_row) + 1,
    int(temp.min_col): int(temp.max_col) + 1
]
plt.imshow(temp, cmap = "grey")

In [ ]:
bbox = bbox.convex_hull.reset_index(drop = False, name = "geometry").set_index("key")

In [ ]:
bbox.geometry.crs